In [ ]:
from __future__ import annotations

import sys
from pathlib import Path


def _find_workspace_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "config" / "workspace.json").exists():
            return candidate
    raise FileNotFoundError("Could not locate workspace root from the current working directory.")


WORKSPACE_DIR = _find_workspace_root(Path.cwd())
PAPER_ROOT = WORKSPACE_DIR.parent
RESEARCH_ROOT = PAPER_ROOT.parent
SHARED_ASSETS_DIR = RESEARCH_ROOT / "shared-assets"
SHARED_CODE_DIR = SHARED_ASSETS_DIR / "code"
if str(SHARED_CODE_DIR) not in sys.path:
    sys.path.insert(0, str(SHARED_CODE_DIR))

from workspace_rooting.workspace_paths import canonical_workspace_paths

paths = canonical_workspace_paths(WORKSPACE_DIR)
WORKSPACE_DIR = paths["workspace"]
CODE_DIR = paths["code"]
CONFIG_DIR = paths["config"]
DATA_DIR = paths["data"]
OUTPUTS_DIR = paths["outputs"]
DOCS_DIR = paths["docs"]
LOCAL_DIR = paths["local"]
SHARED_ASSETS_DIR = paths["shared_assets"]
APPENDIX_DIR = CODE_DIR / "appendix"
APPENDIX_OUTPUTS_DIR = OUTPUTS_DIR / "appendix"
APPENDIX_RUNS_DIR = APPENDIX_OUTPUTS_DIR / "runs"
APPENDIX_FIGURES_DIR = APPENDIX_OUTPUTS_DIR / "figures"
OPENREFINE_DIR = LOCAL_DIR / "curation" / "openrefine"

for candidate in [CODE_DIR, APPENDIX_DIR]:
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

APPENDIX_RUNS_DIR.mkdir(parents=True, exist_ok=True)
APPENDIX_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("workspace_root:", WORKSPACE_DIR)
print("shared_assets_root:", SHARED_ASSETS_DIR)
print("appendix_outputs_root:", APPENDIX_OUTPUTS_DIR)

from run_modes.api_run_mode import (
    RunMode,
    append_jsonl_row,
    rebuild_or_resume_message,
    require_env_var,
    resolve_run_mode,
    should_create_client,
)


# RDF Triplet Extraction (Responses API)

This notebook is a cleaned experimental replacement for the older OpenAI triplet extraction workflow.

## Purpose

- extract constitutive and identity-style triplets about `dark matter`
- use the local paper-ready ADS frame rather than an ad hoc CSV
- use the OpenAI Responses API with Structured Outputs
- keep the run resumable and cache every processed abstract
- flatten results directly into tabular outputs for later normalization and analysis

## Default model

This notebook defaults to `gpt-5-mini`, which is a good fit for a high-volume, well-specified extraction task. If you want a higher-precision spot check, switch `MODEL` to `gpt-5`.


In [ ]:
from __future__ import annotations

import importlib
import json
import os
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

import pandas as pd
import tiktoken
from openai import OpenAI
from tqdm.auto import tqdm

import rdf_triplet_extraction as rte
importlib.reload(rte)

SYSTEM_PROMPT = rte.SYSTEM_PROMPT
TRIPLET_SCHEMA = rte.TRIPLET_SCHEMA
append_jsonl = rte.append_jsonl
ensure_run_dirs = rte.ensure_run_dirs
flatten_triplet_requests = rte.flatten_triplet_requests
load_completed_bibcodes = rte.load_completed_bibcodes
normalize_triplets = rte.normalize_triplets
prepare_input_frame = rte.prepare_input_frame
prompt_hash = rte.prompt_hash
write_run_manifest = rte.write_run_manifest


def rebuild_triplet_outputs_from_cache(paths, input_path, run_id, model, abstract_mode, prompt_sha256,
                                       df_input, seed_sample_df=None, composite_seed_run_id=None,
                                       composite_seed_year_min=None, composite_seed_year_max=None):
    fresh_flat_df = flatten_triplet_requests(paths["requests_jsonl"])
    seed_flat_df = (
        flatten_triplet_requests(paths["seed_requests_jsonl"])
        if paths["seed_requests_jsonl"].exists()
        else pd.DataFrame(columns=fresh_flat_df.columns)
    )

    frames = [df for df in [seed_flat_df, fresh_flat_df] if not df.empty]
    if frames:
        flat_df = pd.concat(frames, ignore_index=True)
    else:
        flat_df = pd.DataFrame(
            columns=["bibcode", "year", "title", "subject", "predicate", "object", "claim_type", "evidence_text"]
        )

    flat_df.to_parquet(paths["flat_parquet"], index=False)
    flat_df.to_csv(paths["flat_csv"], index=False)

    seed_n_rows = 0 if seed_sample_df is None else len(seed_sample_df)
    completed_fresh_rows = len(load_completed_bibcodes(paths["requests_jsonl"]))
    combined_input_rows = len(df_input) + seed_n_rows
    combined_completed_rows = completed_fresh_rows + seed_n_rows

    extra_manifest = {
        "seed_run_id": composite_seed_run_id if seed_n_rows else None,
        "seed_year_min": composite_seed_year_min if seed_n_rows else None,
        "seed_year_max": composite_seed_year_max if seed_n_rows else None,
        "seed_n_rows": seed_n_rows,
        "fresh_n_rows": len(df_input),
    }

    write_run_manifest(
        paths["manifest_json"],
        run_id=run_id,
        model=model,
        abstract_mode=abstract_mode,
        input_path=input_path,
        prompt_sha256=prompt_sha256,
        n_input_rows=combined_input_rows,
        n_completed_rows=combined_completed_rows,
        n_flat_triplets=len(flat_df),
        extra=extra_manifest,
    )
    return flat_df


In [ ]:
# Root resolution now comes from the shared workspace bootstrap cell.

## Configuration

The defaults below aim for a resumable full run.

- `MODEL`: extraction model
- `MAX_WORKERS`: concurrent requests
- `SAMPLE_N`: optional dry-run sample size
- `FORCE_REPROCESS`: whether to ignore the existing request cache
- `ABSTRACT_MODE`: use raw abstracts with only light normalization

For this task, raw-normalized abstracts are preferable to aggressively cleaned abstracts. Cleaning can erase hedges, candidate language, and the local syntax that distinguishes identity claims from weaker constitutive or candidate claims.

When `SAMPLE_N` is set, the notebook now draws a period-stratified adjudication sample rather than simply taking the first `N` rows. This prevents the first-pass review from collapsing into the earliest historical literature only.


In [ ]:
RUN_MODE = 'resume'   # 'live' | 'resume' | 'replay'
MODEL = 'gpt-5-mini'
MAX_WORKERS = 4
MIN_ABSTRACT_CHARS = 80
MIN_TOKENS = 40
ABSTRACT_MODE = 'raw_normalized'
FORCE_REPROCESS = False

# Recommended presets
# Adjudication sample:
# SAMPLE_N = 50
# RUN_ID = 'rdf_triplets__gpt_5_mini__sample__v4'
# SUPPLEMENTARY_FILTER = False

# Composite retained run (10K total):
SAMPLE_N: int | None = None
RUN_ID = 'rdf_triplets__gpt_5_mini__supplementary__10k__v1'
SUPPLEMENTARY_FILTER = True
SUPPLEMENTARY_YEAR_MIN = 1990
SUPPLEMENTARY_CUE_REGEX = r'candidate|viable|as dark matter|dark matter candidate|constitut|compos|consist(?:s|ing)? of|made of|wimp|axion|neutralino|sterile neutrino|hidden photon|primordial black hole|self[ -]interacting dark matter|warm dark matter|cold dark matter|fuzzy dark matter|fermionic dark matter|bosonic dark matter|scalar dark matter'

COMPOSITE_SEED_ENABLED = True
COMPOSITE_SEED_RUN_ID = 'rdf_triplets__gpt_5_mini__supplementary__10k__v1'
COMPOSITE_SEED_YEAR_MIN = 1990
COMPOSITE_SEED_YEAR_MAX = 2000
COMPOSITE_SEED_N = 800
COMPOSITE_RANDOM_STATE = 42
COMPOSITE_FRESH_TARGETS = {
    '2001_2009': 1700,
    '2010_2019': 3500,
    '2020_2029': 4000,
}

EXPERIMENT_ROOT = APPENDIX_OUTPUTS_DIR
INPUT_PATH = DATA_DIR / 'parquet' / 'ADS_MAIN_FRAME_2025.parquet'
RUN_ROOT = APPENDIX_RUNS_DIR / RUN_ID
PATHS = ensure_run_dirs(RUN_ROOT)
PATHS['seed_requests_jsonl'] = RUN_ROOT / 'seed_triplet_requests.jsonl'
PATHS['seed_selection_json'] = RUN_ROOT / 'seed_selection.json'
PROMPT_SHA256 = prompt_hash(SYSTEM_PROMPT, TRIPLET_SCHEMA)

print('INPUT_PATH:', INPUT_PATH)
print('RUN_ROOT:', RUN_ROOT)
print('ABSTRACT_MODE:', ABSTRACT_MODE)
print('SAMPLE_N:', SAMPLE_N)
print('FORCE_REPROCESS:', FORCE_REPROCESS)
print('SUPPLEMENTARY_FILTER:', SUPPLEMENTARY_FILTER)
print('SUPPLEMENTARY_YEAR_MIN:', SUPPLEMENTARY_YEAR_MIN)
print('SUPPLEMENTARY_CUE_REGEX:', SUPPLEMENTARY_CUE_REGEX)
print('COMPOSITE_SEED_ENABLED:', COMPOSITE_SEED_ENABLED)
print('COMPOSITE_SEED_RUN_ID:', COMPOSITE_SEED_RUN_ID)
print('COMPOSITE_FRESH_TARGETS:', COMPOSITE_FRESH_TARGETS)
print('REQUESTS_JSONL:', PATHS['requests_jsonl'])
print('SEED_REQUESTS_JSONL:', PATHS['seed_requests_jsonl'])
print('PROMPT_SHA256:', PROMPT_SHA256[:12], '...')


## Recommended supplementary run settings

Use this notebook in two modes only:

1. `SAMPLE_N = 50` with a versioned `RUN_ID` for adjudication
2. `SAMPLE_N = None` with a fresh `RUN_ID` for the retained composite run

Recommended retained composite settings:

- `MODEL = "gpt-5-mini"`
- `SAMPLE_N = None`
- `RUN_ID = "rdf_triplets__gpt_5_mini__supplementary__10k__v1"`
- `FORCE_REPROCESS = False`
- `ABSTRACT_MODE = "raw_normalized"`
- `SUPPLEMENTARY_FILTER = True`
- `COMPOSITE_SEED_RUN_ID = "rdf_triplets__gpt_5_mini__supplementary__v2"`
- `COMPOSITE_SEED_N = 800` from cached `1990–2000` rows
- `COMPOSITE_FRESH_TARGETS = {"2001_2009": 1700, "2010_2019": 3500, "2020_2029": 4000}`
- `MAX_WORKERS = 4`

Why this is the right default:

- it preserves a random subset of the early rows you already paid for
- it avoids a chronologically biased prefix run
- it keeps the final supplementary corpus at 10,000 abstracts
- it remains cheap enough to be practical for a supplementary analysis
- it writes a combined retained run whose parquet/csv include both the cached seed sample and the fresh stratified sample

After the run finishes, inspect the predicate distribution and manually spot-check a small random slice before treating the output as stable supplementary material.


## Load and prepare the local ADS frame

This uses the paper-local ADS frame already built in `semantic-change/workspace/data/parquet/ADS_MAIN_FRAME_2025.parquet`.

The notebook keeps the raw abstract text and only applies light normalization for prompt use.


In [ ]:
df_raw = pd.read_parquet(INPUT_PATH)
print(df_raw.columns.tolist())
print('rows:', len(df_raw))

df_input = prepare_input_frame(
    df_raw,
    min_chars=MIN_ABSTRACT_CHARS,
    abstract_mode=ABSTRACT_MODE,
).copy()

enc = tiktoken.encoding_for_model(MODEL)
df_input['n_tokens'] = df_input['abstract_prompt'].map(lambda text: len(enc.encode(text)))
df_input = df_input[df_input['n_tokens'] >= MIN_TOKENS].reset_index(drop=True)

if SUPPLEMENTARY_FILTER:
    before_filter_n = len(df_input)
    text_surface = (
        df_input['title'].fillna('') + ' ' + df_input['abstract_prompt'].fillna('')
    ).str.lower().str.replace(r'\s+', ' ', regex=True)
    year_mask = df_input['year'].fillna(0).astype(int) >= SUPPLEMENTARY_YEAR_MIN
    cue_mask = text_surface.str.contains(SUPPLEMENTARY_CUE_REGEX, regex=True, na=False)
    df_input = df_input[year_mask & cue_mask].copy()
    df_input = df_input.reset_index(drop=True)
    print('supplementary filter kept:', len(df_input), 'of', before_filter_n)

SEED_SAMPLE_DF = pd.DataFrame()
SEED_BIBCODES: set[str] = set()
SEED_RUN_DIR = EXPERIMENT_ROOT / 'runs' / COMPOSITE_SEED_RUN_ID
SEED_SOURCE_PATH = SEED_RUN_DIR / 'seed_triplet_requests.jsonl'
if not SEED_SOURCE_PATH.exists():
    SEED_SOURCE_PATH = SEED_RUN_DIR / 'triplet_requests.jsonl'

def assign_period(year):
    if pd.isna(year):
        return 'unknown'
    y = int(year)
    if 1930 <= y <= 1959:
        return '1930_1959'
    if 1960 <= y <= 1989:
        return '1960_1989'
    if 1990 <= y <= 2009:
        return '1990_2009'
    if 2010 <= y <= 2019:
        return '2010_2019'
    if 2020 <= y <= 2029:
        return '2020_2029'
    return 'outside_range'

if SAMPLE_N is not None:
    df_input['period_bin'] = df_input['year'].map(assign_period)
    period_order = ['1930_1959', '1960_1989', '1990_2009', '2010_2019', '2020_2029']
    available_periods = [label for label in period_order if label in set(df_input['period_bin'])]

    if available_periods:
        per_period = max(1, SAMPLE_N // len(available_periods))
        sampled_parts = []
        for label in available_periods:
            subset = df_input[df_input['period_bin'] == label]
            take_n = min(per_period, len(subset))
            sampled_parts.append(subset.sample(take_n, random_state=42))
        sampled = pd.concat(sampled_parts, ignore_index=True)
        if len(sampled) < SAMPLE_N:
            remainder = df_input.loc[~df_input['bibcode'].isin(sampled['bibcode'])]
            extra_n = min(SAMPLE_N - len(sampled), len(remainder))
            if extra_n > 0:
                sampled = pd.concat([sampled, remainder.sample(extra_n, random_state=42)], ignore_index=True)
        df_input = sampled.sort_values(['year', 'bibcode'], kind='mergesort').reset_index(drop=True)
elif COMPOSITE_SEED_ENABLED:
    if not SEED_SOURCE_PATH.exists():
        raise FileNotFoundError(f'Missing cached seed run: {SEED_SOURCE_PATH}')

    seed_rows = []
    with open(SEED_SOURCE_PATH, 'r', encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            row = json.loads(line)
            year = row.get('year')
            if year is None:
                continue
            year = int(year)
            if COMPOSITE_SEED_YEAR_MIN <= year <= COMPOSITE_SEED_YEAR_MAX:
                seed_rows.append(row)

    if len(seed_rows) < COMPOSITE_SEED_N:
        raise ValueError(
            f'Cached seed pool has only {len(seed_rows)} rows; need {COMPOSITE_SEED_N}.'
        )

    seed_index = (
        pd.Series(range(len(seed_rows)))
        .sample(n=COMPOSITE_SEED_N, random_state=COMPOSITE_RANDOM_STATE, replace=False)
        .sort_values()
        .tolist()
    )
    seed_sample_rows = [seed_rows[i] for i in seed_index]
    SEED_SAMPLE_DF = pd.DataFrame(seed_sample_rows).copy()
    SEED_SAMPLE_DF = SEED_SAMPLE_DF.sort_values(['year', 'bibcode'], kind='mergesort').reset_index(drop=True)
    SEED_BIBCODES = set(SEED_SAMPLE_DF['bibcode'].astype(str))

    with open(PATHS['seed_requests_jsonl'], 'w', encoding='utf-8') as f:
        for row in SEED_SAMPLE_DF.to_dict(orient='records'):
            f.write(json.dumps(row, ensure_ascii=False) + '\n')

    seed_selection_meta = {
        'seed_run_id': COMPOSITE_SEED_RUN_ID,
        'seed_year_min': COMPOSITE_SEED_YEAR_MIN,
        'seed_year_max': COMPOSITE_SEED_YEAR_MAX,
        'seed_n_rows': len(SEED_SAMPLE_DF),
        'random_state': COMPOSITE_RANDOM_STATE,
        'fresh_targets': COMPOSITE_FRESH_TARGETS,
    }
    PATHS['seed_selection_json'].write_text(
        json.dumps(seed_selection_meta, ensure_ascii=False, indent=2),
        encoding='utf-8',
    )

    fresh_periods = [
        ('2001_2009', 2001, 2009),
        ('2010_2019', 2010, 2019),
        ('2020_2029', 2020, 2029),
    ]
    df_input['year'] = df_input['year'].astype(int)
    fresh_pool = df_input[
        (df_input['year'] >= 2001)
        & (~df_input['bibcode'].astype(str).isin(SEED_BIBCODES))
    ].copy()
    fresh_pool['period_bin'] = fresh_pool['year'].map(assign_period)

    sampled_parts = []
    sampled_counts = {}
    for label, start, end in fresh_periods:
        target_n = COMPOSITE_FRESH_TARGETS[label]
        subset = fresh_pool[(fresh_pool['year'] >= start) & (fresh_pool['year'] <= end)].copy()
        if len(subset) < target_n:
            raise ValueError(f'Period {label} has only {len(subset)} rows; need {target_n}.')
        sampled = subset.sample(n=target_n, random_state=COMPOSITE_RANDOM_STATE, replace=False)
        sampled_parts.append(sampled)
        sampled_counts[label] = len(sampled)

    df_input = pd.concat(sampled_parts, ignore_index=True)
    df_input = df_input.sample(frac=1, random_state=COMPOSITE_RANDOM_STATE).reset_index(drop=True)

    print('cached seed rows:', len(SEED_SAMPLE_DF))
    print('cached seed years:', COMPOSITE_SEED_YEAR_MIN, '-', COMPOSITE_SEED_YEAR_MAX)
    print('fresh period targets:', sampled_counts)
    print('combined target rows:', len(SEED_SAMPLE_DF) + len(df_input))

print('prepared rows:', len(df_input))
if 'period_bin' in df_input.columns:
    print(df_input['period_bin'].value_counts().sort_index())
if not SEED_SAMPLE_DF.empty:
    print('seed sample preview:')
    print(SEED_SAMPLE_DF[['bibcode', 'year', 'title']].head(5))
print(df_input[['bibcode', 'year', 'title', 'n_tokens']].head(10))


## API client

The notebook expects `OPENAI_API_KEY` in the environment.

Example:

```bash
export OPENAI_API_KEY="..."
```


In [ ]:
assert os.getenv('OPENAI_API_KEY'), 'Set OPENAI_API_KEY in the environment before running the extraction.'
print('OPENAI_API_KEY found in environment.')


In [ ]:
state = resolve_run_mode(
    run_mode=RUN_MODE,
    cache_path=PATHS['requests_jsonl'],
    expected_ids=df_input['bibcode'].astype(str).tolist(),
    completed_id_fn=lambda rows: {str(r['bibcode']) for r in rows},
)
print(rebuild_or_resume_message(state))

if state.mode is not RunMode.REPLAY:
    require_env_var('OPENAI_API_KEY')
    client = OpenAI() if should_create_client(state) else None
    print('OpenAI client ready.' if client is not None else 'No pending rows; client not created.')


## Structured extraction helpers

The extraction is intentionally conservative and uses Structured Outputs so each response must satisfy the schema.

The schema also normalizes predicates into a small controlled vocabulary:

- `is`
- `constitutes`
- `composed_of`
- `candidate_for`


In [ ]:
def build_user_prompt(row: pd.Series) -> str:
    title = str(row.get('title') or '').strip()
    abstract = str(row.get('abstract_prompt') or '').strip()
    bibcode = str(row.get('bibcode') or '').strip()
    year = str(row.get('year') or '').strip()

    return (
        f"Bibcode: {bibcode}\n"
        f"Year: {year}\n"
        f"Title: {title}\n\n"
        "Abstract:\n"
        f"{abstract}\n\n"
        "Return only explicit constitutive, identity-style, or candidate-style triplets about dark matter. "
        "If the abstract contains no such explicit claim, return an empty triplet list."
    )


def call_triplet_extraction(row: pd.Series) -> dict:
    response = client.responses.create(
        model=MODEL,
        input=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_user_prompt(row)},
        ],
        text={
            'format': {
                'type': 'json_schema',
                'name': 'dark_matter_triplets',
                'schema': TRIPLET_SCHEMA,
                'strict': True,
            }
        },
    )
    parsed = json.loads(response.output_text)
    triplets, filter_notes = normalize_triplets(parsed.get('triplets', []))
    return {
        'bibcode': str(row['bibcode']),
        'year': int(row['year']) if pd.notna(row['year']) else None,
        'title': str(row.get('title') or ''),
        'model': MODEL,
        'abstract_mode': ABSTRACT_MODE,
        'prompt_sha256': PROMPT_SHA256,
        'triplets': triplets,
        'notes': parsed.get('notes', []) + filter_notes,
        'response_id': getattr(response, 'id', None),
    }


## Resumable extraction run

This cell appends one JSON object per processed abstract to `triplet_requests.jsonl`.


In [ ]:
if state.mode is RunMode.REPLAY:
    flat_df = rebuild_triplet_outputs_from_cache(
        PATHS,
        INPUT_PATH,
        RUN_ID,
        MODEL,
        ABSTRACT_MODE,
        PROMPT_SHA256,
        df_input,
        seed_sample_df=SEED_SAMPLE_DF,
        composite_seed_run_id=COMPOSITE_SEED_RUN_ID,
        composite_seed_year_min=COMPOSITE_SEED_YEAR_MIN,
        composite_seed_year_max=COMPOSITE_SEED_YEAR_MAX,
    )
    print('Rebuilt outputs from cache only.')
    display(flat_df.head(10))
else:
    queue_df = df_input[df_input['bibcode'].astype(str).isin(state.pending_ids)].copy()

    print('already completed:', state.completed_n)
    print('queued rows:', len(queue_df))

    results = []
    errors = []

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        futures = {ex.submit(call_triplet_extraction, row): row for _, row in queue_df.iterrows()}
        for fut in tqdm(as_completed(futures), total=len(futures)):
            row = futures[fut]
            try:
                payload = fut.result()
                append_jsonl_row(PATHS['requests_jsonl'], payload)
                results.append(payload)
            except Exception as exc:
                errors.append({'bibcode': str(row['bibcode']), 'error': repr(exc)})

    flat_df = rebuild_triplet_outputs_from_cache(
        PATHS,
        INPUT_PATH,
        RUN_ID,
        MODEL,
        ABSTRACT_MODE,
        PROMPT_SHA256,
        df_input,
        seed_sample_df=SEED_SAMPLE_DF,
        composite_seed_run_id=COMPOSITE_SEED_RUN_ID,
        composite_seed_year_min=COMPOSITE_SEED_YEAR_MIN,
        composite_seed_year_max=COMPOSITE_SEED_YEAR_MAX,
    )

    print('new results:', len(results))
    print('errors:', len(errors))
    if errors:
        print(errors[:5])
    display(flat_df.head(10))


In [ ]:
fresh_flat_df = flatten_triplet_requests(PATHS['requests_jsonl'])
seed_flat_df = (
    flatten_triplet_requests(PATHS['seed_requests_jsonl'])
    if PATHS['seed_requests_jsonl'].exists()
    else pd.DataFrame(columns=fresh_flat_df.columns)
)

frames = [df for df in [seed_flat_df, fresh_flat_df] if not df.empty]
if frames:
    flat_df = pd.concat(frames, ignore_index=True)
else:
    flat_df = pd.DataFrame(columns=['bibcode', 'year', 'title', 'subject', 'predicate', 'object', 'claim_type', 'evidence_text'])

flat_df.to_parquet(PATHS['flat_parquet'], index=False)
flat_df.to_csv(PATHS['flat_csv'], index=False)

seed_n_rows = len(SEED_SAMPLE_DF) if 'SEED_SAMPLE_DF' in globals() else 0
completed_fresh_rows = len(load_completed_bibcodes(PATHS['requests_jsonl']))
combined_input_rows = len(df_input) + seed_n_rows
combined_completed_rows = completed_fresh_rows + seed_n_rows
extra_manifest = {
    'seed_run_id': COMPOSITE_SEED_RUN_ID if seed_n_rows else None,
    'seed_year_min': COMPOSITE_SEED_YEAR_MIN if seed_n_rows else None,
    'seed_year_max': COMPOSITE_SEED_YEAR_MAX if seed_n_rows else None,
    'seed_n_rows': seed_n_rows,
    'fresh_n_rows': len(df_input),
}

write_run_manifest(
    PATHS['manifest_json'],
    run_id=RUN_ID,
    model=MODEL,
    abstract_mode=ABSTRACT_MODE,
    input_path=INPUT_PATH,
    prompt_sha256=PROMPT_SHA256,
    n_input_rows=combined_input_rows,
    n_completed_rows=combined_completed_rows,
    n_flat_triplets=len(flat_df),
    extra=extra_manifest,
)

print('seed flat rows:', len(seed_flat_df))
print('fresh flat rows:', len(fresh_flat_df))
print('combined flat rows:', len(flat_df))
print('flat parquet:', PATHS['flat_parquet'])
print('manifest:', PATHS['manifest_json'])
flat_df.head(10)


In [ ]:
fresh_flat_df = flatten_triplet_requests(PATHS['requests_jsonl'])
seed_flat_df = (
    flatten_triplet_requests(PATHS['seed_requests_jsonl'])
    if PATHS['seed_requests_jsonl'].exists()
    else pd.DataFrame(columns=fresh_flat_df.columns)
)

frames = [df for df in [seed_flat_df, fresh_flat_df] if not df.empty]
if frames:
    flat_df = pd.concat(frames, ignore_index=True)
else:
    flat_df = pd.DataFrame(columns=['bibcode', 'year', 'title', 'subject', 'predicate', 'object', 'claim_type', 'evidence_text'])

flat_df.to_parquet(PATHS['flat_parquet'], index=False)
flat_df.to_csv(PATHS['flat_csv'], index=False)

seed_n_rows = len(SEED_SAMPLE_DF) if 'SEED_SAMPLE_DF' in globals() else 0
completed_fresh_rows = len(load_completed_bibcodes(PATHS['requests_jsonl']))
combined_input_rows = len(df_input) + seed_n_rows
combined_completed_rows = completed_fresh_rows + seed_n_rows
extra_manifest = {
    'seed_run_id': COMPOSITE_SEED_RUN_ID if seed_n_rows else None,
    'seed_year_min': COMPOSITE_SEED_YEAR_MIN if seed_n_rows else None,
    'seed_year_max': COMPOSITE_SEED_YEAR_MAX if seed_n_rows else None,
    'seed_n_rows': seed_n_rows,
    'fresh_n_rows': len(df_input),
}

write_run_manifest(
    PATHS['manifest_json'],
    run_id=RUN_ID,
    model=MODEL,
    abstract_mode=ABSTRACT_MODE,
    input_path=INPUT_PATH,
    prompt_sha256=PROMPT_SHA256,
    n_input_rows=combined_input_rows,
    n_completed_rows=combined_completed_rows,
    n_flat_triplets=len(flat_df),
    extra=extra_manifest,
)

print('seed flat rows:', len(seed_flat_df))
print('fresh flat rows:', len(fresh_flat_df))
print('combined flat rows:', len(flat_df))
print('flat parquet:', PATHS['flat_parquet'])
print('manifest:', PATHS['manifest_json'])
flat_df.head(10)


## Quick inspection

These checks make it easier to see whether the model is behaving conservatively enough before scaling up.


In [ ]:
if len(flat_df):
    print(flat_df['predicate'].value_counts(dropna=False).head(20))
    display(flat_df.sample(min(10, len(flat_df)), random_state=42))
else:
    print('No triplets extracted yet.')


## Recommended next steps

- inspect a small sample first with `SAMPLE_N = 25`
- if output quality is good, set `SAMPLE_N = None`
- if precision is still too loose, keep the same schema and tighten only the prompt
- if you want a more expensive precision pass, rerun with `MODEL = "gpt-5"`
